## 1.set up the database 

In [1]:
import os
import sqlite3
import pandas as pd

CUSTOM_DB_PATH = "sqllite.db"

if not os.path.exists(CUSTOM_DB_PATH):
    raise FileNotFoundError(
        "sqllite.db was not found. Make sure the database file is uploaded."
    )

conn = sqlite3.connect(CUSTOM_DB_PATH)

print(f"Connected to: {CUSTOM_DB_PATH}")

Connected to: sqllite.db


In [2]:
def run_query(query):
    """Execute a SQL query and return the result as a DataFrame."""
    
    cleaned = "\n".join(
        line for line in query.splitlines()
        if not line.strip().startswith("--")
    ).strip()

    if not cleaned:
        print("Write your SQL query in the cell first.")
        return None

    return pd.read_sql_query(cleaned, conn)

In [4]:
def list_tables(connection=conn):
    return pd.read_sql_query(
        """
        SELECT name AS table_name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name;
        """,
        connection
    )

In [5]:
list_tables()

,table_name
0,p1_applications
1,p1_bureau_summary
2,p1_funded_loans
3,p1_payroll_monthly
4,p2_order_items
5,p2_orders
6,p2_returns
7,p2_users
8,p3_billing_monthly
9,p3_qos_daily


In [6]:
run_query("""
select *
from p4_driver_pings
limit 3
""")

,driver_id,ts,lat,lon
0,900433,2024-10-10 18:00:36,25.257844,55.250156
1,900096,2025-02-07 03:00:49,30.063390,31.234836
2,900886,2024-11-10 07:37:24,30.088119,31.238852


In [8]:
def show_schema(table_name, connection=conn):
    safe_name = table_name.replace("'", "''")
    
    return pd.read_sql_query(
        f"""
        PRAGMA table_info('{safe_name}');
        """,
        connection
    )

In [10]:
show_schema("p4_driver_pings")

,cid,name,type,notnull,dflt_value,pk
0,0,driver_id,INTEGER,0,None,0
1,1,ts,TIMESTAMP,0,None,0
2,2,lat,REAL,0,None,0
3,3,lon,REAL,0,None,0


## Question 1
**""Which cities have the highest ride cancellation rates?""**


In [11]:
show_schema ("p4_rides")

,cid,name,type,notnull,dflt_value,pk
0,0,ride_id,INTEGER,0,None,0
1,1,request_ts,TIMESTAMP,0,None,0
2,2,pickup_lat,REAL,0,None,0
3,3,pickup_lon,REAL,0,None,0
4,4,drop_lat,REAL,0,None,0
5,5,drop_lon,REAL,0,None,0
6,6,user_id,INTEGER,0,None,0
7,7,driver_id,INTEGER,0,None,0
8,8,surge_mult,REAL,0,None,0
9,9,status,TEXT,0,None,0


In [13]:
show_schema ("p4_drivers")

,cid,name,type,notnull,dflt_value,pk
0,0,driver_id,INTEGER,0,None,0
1,1,city,TEXT,0,None,0
2,2,signup_dt,TIMESTAMP,0,None,0
3,3,status,TEXT,0,None,0


In [14]:
run_query("""
 select status from p4_rides
""")

,status
0,completed
1,completed
2,completed
3,cancelled
4,completed
...,...
19995,completed
19996,completed
19997,completed
19998,completed


In [132]:
run_query("""
  select d.city	as highest_city , count( case when r.status ='cancelled' then 1 end) as cancelled_rides_count ,
  count( r.ride_id ) as total_rides , 
  (count( case when r.status ='cancelled' then 1 end) *100.00 ) / count( r.status ) as cancelled_rides_rates
  from p4_rides as r
  inner join p4_drivers as d
  on r.driver_id = d.driver_id
  group by d.city
  order by  cancelled_rides_rates  desc
  
""")

,highest_city,cancelled_rides_count,total_rides,cancelled_rides_rates
0,Jeddah,513,4009,12.796209
1,Cairo,1001,7935,12.614997
2,Riyadh,627,5035,12.452830
3,Dubai,341,3021,11.287653


## 2.Question
**""Does rain increase cancellation rates compared to dry weather? ""**


In [13]:
show_schema ("p4_weather")

,cid,name,type,notnull,dflt_value,pk
0,0,city,TEXT,0,None,0
1,1,dt,TIMESTAMP,0,None,0
2,2,rain_mm,REAL,0,None,0
3,3,temp_c,REAL,0,None,0
4,4,wind_kph,REAL,0,None,0


In [14]:
run_query("""
  select ( case when w. rain_mm >0 then 'rain'
   else 'dry' 
   end) as weather,count( case when r.status ='cancelled' then 1 end) as cancelation_rides_count  , count ( ride_id	) as total_rides ,
round((count( case when r.status ='cancelled' then 1 end) *100.00 ) / count( r.status ) ,2 )as cancelled_rides_rates
  from p4_rides as r
  inner join p4_drivers as d
  on r.driver_id = d.driver_id
  inner join p4_weather as w
  on d.city	= w.city  
  and strftime('%Y-%m-%d %H', r.request_ts) = strftime('%Y-%m-%d %H', w.dt)
  group by weather
  order by cancelled_rides_rates desc 
  
""")

,weather,cancelation_rides_count,total_rides,cancelled_rides_rates
0,rain,6,35,17.14
1,dry,106,812,13.05


### yes , it depend on it 


# 3.Question
**At what times of day is surge pricing highest?**


In [19]:
run_query("""
 select request_ts	from p4_rides
""")

,request_ts
0,2024-10-10 18:03:34
1,2025-02-07 03:01:45
2,2024-11-10 07:39:16
3,2025-04-02 05:49:30
4,2025-01-28 03:34:05
...,...
19995,2025-01-13 21:07:08
19996,2025-01-08 15:19:50
19997,2025-06-18 02:42:55
19998,2025-04-02 16:22:17


In [43]:
run_query("""
 select surge_mult  from p4_rides
""")

,surge_mult
0,1.33
1,1.03
2,1.19
3,1.00
4,1.00
...,...
19995,1.11
19996,1.03
19997,1.03
19998,1.00


In [20]:
run_query ("""
 select *
 from p4_rides as r

""")

,ride_id,request_ts,pickup_lat,pickup_lon,drop_lat,drop_lon,user_id,driver_id,surge_mult,status,cancel_reason,eta_req_s,trip_s,fare_usd
0,950001,2024-10-10 18:03:34,25.261517,55.250104,25.228292,55.288885,400001,900433,1.33,completed,NaN,350,998,5.25
1,950002,2025-02-07 03:01:45,30.068378,31.241710,30.043518,31.301634,400002,900096,1.03,completed,NaN,230,1032,4.15
2,950003,2024-11-10 07:39:16,30.090041,31.230649,30.016879,31.245143,400003,900886,1.19,completed,NaN,181,1508,5.92
3,950004,2025-04-02 05:49:30,24.732325,46.658631,24.743567,46.696252,400004,900480,1.00,cancelled,No driver,284,628,2.98
4,950005,2025-01-28 03:34:05,21.472845,39.161244,21.498840,39.162097,400005,900793,1.00,completed,NaN,331,391,2.62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,969996,2025-01-13 21:07:08,24.735247,46.713433,24.727782,46.709855,419996,900969,1.11,completed,NaN,170,180,1.75
19996,969997,2025-01-08 15:19:50,30.036521,31.228467,30.041806,31.267240,419997,900454,1.03,completed,NaN,280,586,3.21
19997,969998,2025-06-18 02:42:55,30.083878,31.185834,29.987225,31.186548,419998,901050,1.03,completed,NaN,310,1586,7.29
19998,969999,2025-04-02 16:22:17,21.481946,39.199074,21.529428,39.172541,419999,900774,1.00,completed,NaN,122,842,4.41


In [41]:
run_query ("""
 select (case when cast( strftime('%H',  r.request_ts) as integer )<= 12 then ' morning'
              when  cast(strftime('%H',  r.request_ts) as integer) >12 and  cast(strftime('%H',  r.request_ts) as integer) <= 17  then ' afternoon '
              when cast (strftime('%H',  r.request_ts) as integer )>17 then ' evening'   end )  as   at_periodsoftime , round ( avg( surge_mult ),2) as averge_surge 
 from p4_rides as r
 group by  at_periodsoftime
 order by averge_surge  desc 

""")

,at_periodsoftime,averge_surge
0,evening,1.09
1,morning,1.07
2,afternoon,1.06


**another solution**

In [32]:
run_query ("""
 select  strftime('%H',  r.request_ts)  as   at_hours , round ( avg( surge_mult ),2) as averge_surge 
 from p4_rides as r
 group by at_hours
 order by averge_surge  desc 
 limit 1
""")

,at_hours,averge_surge
0,19,1.21


# 4.Question
**Do surge rides have longer ETAs (pickup times)?**


In [45]:
run_query ("""
 select (case when surge_mult >1 then ' surge_rides' 
              else ' not_surge_rides'  end ) as rides , round (avg(eta_req_s),2) as avg_pickup_times
  from p4_rides
  group by  rides
  order by avg_pickup_times desc 
""")

,rides,avg_pickup_times
0,surge_rides,270.37
1,not_surge_rides,238.67


### yes , it has longer pickuptimes 

# 5.Question
**How does average fare change with surge levels across cities?**


In [64]:
run_query("""
 select  round (avg(r.fare_usd),2) as  average_fare ,r. surge_mult as surge_levels , d.city as cities
 from p4_rides as r
 inner join p4_drivers as d
  on r.driver_id = d.driver_id
  group by surge_levels ,cities
  order by average_fare desc , surge_levels , cities
""")

,average_fare,surge_levels,cities
0,9.32,1.13,Riyadh
1,7.37,1.24,Riyadh
2,7.04,1.19,Dubai
3,7.02,1.23,Riyadh
4,6.64,1.20,Jeddah
...,...,...,...
64,2.66,1.12,Riyadh
65,2.60,1.16,Jeddah
66,2.49,1.26,Dubai
67,2.46,1.09,Riyadh


# 6.Question
**Are trips longer when it rains?**


In [62]:
 run_query(""" select rain_mm from p4_weather """)

,rain_mm
0,0.00
1,0.00
2,0.00
3,0.00
4,0.00
...,...
1211,0.39
1212,0.00
1213,0.00
1214,0.00


In [66]:
run_query(""" 
 select (case when w.rain_mm >0.00 then 'rainy'
              else ' dry' end )as weather
 , round ( avg(r.trip_s)/60 ,2) as averge_trip_duration_in_min
  from p4_rides as r
  inner join p4_drivers as d
  on r.driver_id = d.driver_id
  inner join p4_weather as w
  on d.city	= w.city  
  and strftime('%Y-%m-%d %H', r.request_ts) = strftime('%Y-%m-%d %H', w.dt)
  group by  weather
order by averge_trip_duration_in_min desc 

""")

,weather,averge_trip_duration_in_min
0,rainy,15.25
1,dry,14.83


### yes, it was longer when it rains 

# 7.Question
**Who are the top 20 drivers by ride volume?**


In [68]:
run_query("""
 select  count ( case when r.status ='completed' then 1 end ) as completed_rides ,  d.driver_id
  from p4_rides as r
  inner join p4_drivers as d
  on r.driver_id = d.driver_id
  group by d.driver_id
  order by  completed_rides desc
  limit 20
""")

,completed_rides,driver_id
0,28,900440
1,28,900955
2,27,900292
3,26,900055
4,26,900795
5,25,900021
6,25,900025
7,25,900063
8,25,900074
9,25,900300


# 8.Question
**Who are the top 20 drivers by revenue earned?**

In [75]:
run_query("""
 select d.driver_id , round( sum(r.fare_usd ) ,2) as total_revenue_inUSD 
 from p4_rides as r
  inner join p4_drivers as d
  on r.driver_id = d.driver_id
  where r.status='completed'
  group by d.driver_id
  order by total_revenue_inUSD
  limit 20
""")

,driver_id,total_revenue_inUSD
0,901013,22.07
1,900471,23.85
2,900200,23.87
3,900784,24.37
4,900583,24.53
5,901170,24.81
6,900781,24.86
7,901194,26.02
8,900809,26.17
9,900011,26.86


# 9.Question
**What hours of the day see the most trips?**

In [77]:
  run_query("""
  select  strftime('%H',  r.request_ts)  as   hours ,count( r.ride_id) as rides_count 
  from  p4_rides as r
  group by   hours
  order by rides_count  desc
  limit 5


  """)

,hours,rides_count
0,06,896
1,21,881
2,20,881
3,18,865
4,17,864


# 10.Question
**How do fares differ for cancelled vs completed rides?**


In [79]:
run_query("""
select count ( case when status not in ('completed' ,'cancelled') then 1 end )
from  p4_rides 




""")

,"count ( case when status not in ('completed' ,'cancelled') then 1 end )"
0,0


In [118]:
run_query("""
select  count(*) as total_rides , round(avg ( fare_usd ),2) as averge_fare ,status
from  p4_rides 
group by status
order by averge_fare desc 
""")

,total_rides,averge_fare,status
0,2482,4.53,cancelled
1,17518,4.50,completed


# 11.Question
**What are the most common reasons for cancellations?**


In [91]:
run_query("""
 select count (cancel_reason) as cancel_reason_count , cancel_reason
 from p4_rides
 where  cancel_reason is not null
 group by  cancel_reason
""")

,cancel_reason_count,cancel_reason
0,502,Driver cancel
1,383,Long ETA
2,600,No driver
3,384,Price high
4,613,Rider cancel


# 12.Question
**Does ETA increase in rainy weather?**


In [131]:
run_query("""
  
select  case when w.rain_mm >0 then 'Rain' else 'Dry' END as weather_condition , avg (r.eta_req_s)  as avg_eta , count(*) as total_rides 
from p4_rides r
join p4_drivers d
on r.driver_id = d.driver_id
join p4_weather w
on d.city = w.city and  strftime('%Y-%m-%d %H', r.request_ts) = strftime('%Y-%m-%d %H', w.dt)
group by weather_condition
order by avg_eta desc ;


""")

,weather_condition,avg_eta,total_rides
0,Rain,329.942857,35
1,Dry,239.853448,812


### yes,the rain has an impact on the estimated time  to picked up time 

# 13.Question
**What % of rides have surge pricing?**


In [121]:
run_query("""
select count (ride_id) as total_rides , count (case when surge_mult >1 then 1 end ) as surge_rides ,
( count (case when surge_mult >1 then 1 end ) *100.0/count (ride_id) ) as surge_rides_percentage 
from p4_rides 

""")

,total_rides,surge_rides,surge_rides_percentage
0,20000,12353,61.765


# Question 14
**Do high winds (>20 kph) make people cancel more often?**


In [125]:
 run_query("""
  select ( case when w.wind_kph > 20 then 'high_winds' 
                else 'low_winds' end) as wind_per_day , round((count( case when r.status='cancelled' then 1 end)*100.0)/count(r.ride_id) ,2)as cancelled_rides_rates
  from p4_rides as r
  inner join p4_drivers as d
  on r.driver_id = d.driver_id
  inner join p4_weather as w
  on d.city	= w.city  
  and strftime('%Y-%m-%d %H', r.request_ts) = strftime('%Y-%m-%d %H', w.dt)
  group by  wind_per_day
  order by cancelled_rides_rates desc
 
 """)

,wind_per_day,cancelled_rides_rates
0,low_winds,13.74
1,high_winds,2.56


### no, high winds does not affect on cancellection rates 

# Question 15
**Do newer drivers complete fewer rides than older ones?**


In [114]:
run_query ("""

select (case when d.signup_dt < '2025-01-01'then 'Older' ELSE 'Newer' end ) as  driver_state,
count (*) AS total_rides,sum (case  when  r.status = 'completed' then  1  end ) as  completed_rides,
round (100*sum(case  when r.status = 'completed' then  1 end ) / count (*), 2) as  completion_rate
from  p4_rides  as r
inner join p4_drivers as  d 
on  r.driver_id = d.driver_id
group by driver_state
order by  completion_rate desc ;


""")

,driver_state,total_rides,completed_rides,completion_rate
0,Newer,9107,8024,88.0
1,Older,10893,9494,87.0


**another solution**

In [111]:
run_query("""
with drivers_rides as(
 select d.driver_id ,d.signup_dt as signup_date ,count (case when r.status ='completed' then 1 end) as completed_rides_per_driver 
 from p4_drivers as d
 left join  p4_rides as r
  on r.driver_id = d.driver_id
  group by d.driver_id , signup_date
  order by signup_date 
),
cutoff as(
select signup_date 
from drivers_rides
limit 1 offset ((select count(*) from drivers_rides )/2 )

)

select (case  when signup_date >= (select signup_date from cutoff ) then 'newdrivers' 
              else 'olddrivers' end ) as drivers_type ,count(*) as total_drivers_count ,
round(avg ( completed_rides_per_driver ),2) as avg_completed_rides
from drivers_rides
group by drivers_type
order by avg_completed_rides desc



 
""")

,drivers_type,total_drivers_count,avg_completed_rides
0,newdrivers,600,14.65
1,olddrivers,600,14.54


### no, the newdrivers completed  rides more than the old drivers(not big diff)